In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

# Profil disciplinaire

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [ ]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")


In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(['q24_research_fields']):
    gb_data, df_rf = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total_freq", y=col, data=gb_data,
                label="Non", color="b", ax=ax)
    sns.barplot(x="freq", y=col, data=gb_data,
                label="Oui", color="r", ax=ax)
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax.yaxis.set_label_text("")
    ax.xaxis.set_label_text("")
    ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/multiple_choice_1.png", bbox_inches='tight', dpi = 200)

In [ ]:
df_rf = df_exp[["q45_clé", 'q24_research_fields']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_domain.value_counts()

## Reduce dimension for research fields
df_rf1

In [ ]:
df_rf1 = df_rf.pivot(index =['q24_research_fields','q24_research_domain'], columns= 'q45_clé', values = "value").fillna(0).reset_index()



In [ ]:
research_fields = df_rf1[df_rf1.columns[2:]].values
research_fields

In [ ]:
import umap
from sklearn.datasets import load_digits

fig, ax = plt.subplots(1, figsize=(10,10))



embedding = umap.UMAP(n_neighbors=2,
                      min_dist=0.1,
                      metric='cosine', random_state = 42).fit_transform(research_fields)

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [x for x in df_rf1.q24_research_domain])

In [ ]:
import umap
import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [ ]:
kmeans_labels = cluster.KMeans(n_clusters=6).fit_predict(embedding)
len(set(kmeans_labels))

In [ ]:
dict_clusters = {}
for n, x in enumerate(df_rf1.q24_research_fields):
    dict_clusters[x] = kmeans_labels[n]


for x in range(len(set(kmeans_labels))):
    print("Cluster :", x)
    for v in dict_clusters:
        if dict_clusters[v] == x:
            print(v)

In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=1,
    min_cluster_size=2, gen_min_span_tree=True
).fit(embedding)


In [ ]:
len(labels.labels_)

In [ ]:
dict_clusters = {}
for n, x in enumerate(df_rf1.q24_research_fields):
    dict_clusters[x] = labels.labels_[n]


for x in range(len(set(labels.labels_))):
    print("Cluster :", x)
    for v in dict_clusters:
        if dict_clusters[v] == x:
            print(v)

In [ ]:
c_palette = ["red", "blue", "green", "yellow", "black", "pink", "purple"]

In [ ]:
fig, ax = plt.subplots(1, figsize=(14, 14))
sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [x for x in labels.labels_], palette=c_palette)

In [ ]:
fig, ax = plt.subplots(1, figsize=(14, 14))
ax.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=[sns.color_palette()[x] for x in df_rf1.q24_research_domain.map({"Sciences sociales et humaines":0, "Sciences de la vie":1, "Sciences physiques et ingénierie":2})])
fig.legend()
plt.gca().set_aspect('equal', 'datalim')

plt.title('UMAP projection of the Penguin dataset', fontsize=24)


In [ ]:
embedding = umap.UMAP(n_neighbors=5,
                      min_dist=0.3,
                      metric='cosine').fit_transform(sc_fields_array.T)

In [ ]:


penguins = pd.read_csv("https://github.com/allisonhorst/palmerpenguins/raw/5b5891f01b52ae26ad8cb9755ec93672f49328a8/data/penguins_size.csv")
penguins.head()



In [ ]:
penguins = penguins.dropna()
penguins.species_short.value_counts()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
reducer = umap.UMAP()

In [ ]:
penguin_data = penguins[
    [
        "culmen_length_mm",
        "culmen_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
    ]
].values
penguin_data
scaled_penguin_data = StandardScaler().fit_transform(penguin_data)


In [ ]:
embedding = reducer.fit_transform(scaled_penguin_data)
embedding.shape

In [ ]:
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=[sns.color_palette()[x] for x in penguins.species_short.map({"Adelie":0, "Chinstrap":1, "Gentoo":2})])
plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP projection of the Penguin dataset', fontsize=24)